# QLoRA Fine-tuning of Llama-3-8B-Instruct for Dialogue Summarization

Fine-tune `meta-llama/Meta-Llama-3-8B-Instruct` on a 2,000-sample subset of DialogSum with QLoRA
(4-bit NF4 quantization + LoRA), on a single Kaggle T4 GPU (15 GB).

**Kaggle settings:** Accelerator = GPU T4 x2 (only GPU 0 is used), Internet = On, Secret `HF_TOKEN` attached.

## 1. Environment

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use only GPU 0 (4-bit model cannot be replicated by DataParallel)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduce memory fragmentation

In [ ]:
!pip install -q "transformers==4.56.1" "accelerate==1.10.1" "peft==0.17.1" \
    "bitsandbytes>=0.48.0" "trl==0.23.0" "datasets==4.1.0" rouge_score

In [ ]:
!nvidia-smi

## 2. Download the model

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

login(UserSecretsClient().get_secret("HF_TOKEN"))

model_path = snapshot_download(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    local_dir="/kaggle/temp/Meta-Llama-3-8B-Instruct",
    ignore_patterns=["original/*"],   # skip the duplicate original-format weights
)
!ls -lh {model_path}

In [ ]:
import gc
import json
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from functools import partial
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model
from rouge_score import rouge_scorer

seed = 42
set_seed(seed)   # fix randomness so results are reproducible

## 3. Data

In [ ]:
dataset = load_dataset("neil-code/dialogsum-test")
print(dataset)
dataset["train"][0]

In [ ]:
# Token length analysis (used to decide max_length)
tok = AutoTokenizer.from_pretrained(model_path)
for split in ["train", "validation", "test"]:
    d = dataset[split]
    dialogue_lens = [len(tok(x).input_ids) for x in d["dialogue"]]
    summary_lens = [len(tok(x).input_ids) for x in d["summary"]]
    print(f"{split:10s} n={len(d):4d} | "
          f"dialogue tokens: mean={np.mean(dialogue_lens):.0f}, max={max(dialogue_lens)} | "
          f"summary tokens: mean={np.mean(summary_lens):.0f}, max={max(summary_lens)}")

In [ ]:
INTRO_BLURB = "Instruct: Below is an instruction that describes a task. Write a response that appropriately completes the request."
INSTRUCTION_KEY = "Input: Please Summarize the below conversation."
RESPONSE_KEY = "Output:"
EOS = "<|eot_id|>"   # added explicitly: Llama-3's tokenizer ignores add_eos_token

def make_prompt(dialogue):
    # Prompt used at inference time (same format as training, without the answer)
    return "\n\n".join([f"\n{INTRO_BLURB}", INSTRUCTION_KEY, dialogue, f"{RESPONSE_KEY}\n"])

def create_prompt_formats(sample):
    # Training text = prompt + reference summary + EOS
    sample["text"] = make_prompt(sample["dialogue"]) + sample["summary"] + EOS
    return sample

print(create_prompt_formats(dataset["train"][0])["text"])

## 4. Load the 4-bit quantized model

In [ ]:
compute_dtype = torch.float16   # T4 does not support bf16

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # store weights in 4 bits
    bnb_4bit_quant_type="nf4",              # NormalFloat4: levels dense around 0
    bnb_4bit_compute_dtype=compute_dtype,   # dequantize to fp16 for computation
    bnb_4bit_use_double_quant=True,         # also quantize the scaling constants
)

original_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    quantization_config=quant_config,
)
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="right")
tokenizer.pad_token = "<|end_of_text|>"   # use a pad token different from EOS (<|eot_id|>)

# Check that training texts end with EOS
ids = tokenizer(create_prompt_formats(dataset["train"][0])["text"]).input_ids
print(tokenizer.convert_ids_to_tokens(ids)[-3:])
print("Ends with EOS:", ids[-1] == tokenizer.eos_token_id)
print("PAD differs from EOS:", tokenizer.pad_token_id != tokenizer.eos_token_id)

## 5. Evaluation helpers and baseline

In [ ]:
eval_tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")
eval_tokenizer.pad_token = "<|end_of_text|>"

def batch_gen(model, prompts, maxlen=150, bs=4):
    # Generate summaries in batches (greedy decoding for reproducible results)
    outputs = []
    for i in range(0, len(prompts), bs):
        toks = eval_tokenizer(prompts[i:i+bs], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(
                **toks,
                max_new_tokens=maxlen,
                do_sample=False,
                eos_token_id=eval_tokenizer.eos_token_id,
                pad_token_id=eval_tokenizer.pad_token_id,
            )
        new_tokens = out[:, toks["input_ids"].shape[1]:]   # keep only newly generated tokens
        outputs += eval_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [o.strip() for o in outputs]

N_EVAL = 100
eval_set = dataset["test"].select(range(N_EVAL))
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def evaluate_rouge(model):
    preds = batch_gen(model, [make_prompt(d) for d in eval_set["dialogue"]])
    scores = [scorer.score(ref, pred) for ref, pred in zip(eval_set["summary"], preds)]
    result = {k: round(float(np.mean([s[k].fmeasure for s in scores])), 4)
              for k in ["rouge1", "rouge2", "rougeL"]}
    return result, preds

In [ ]:
baseline_scores, baseline_preds = evaluate_rouge(original_model)
print("Baseline ROUGE:", baseline_scores)

gc.collect()
torch.cuda.empty_cache()   # free memory left over from generation before training

## 6. Preprocessing

In [ ]:
max_length = 1024   # based on the length analysis: almost all samples are shorter

def preprocess_batch(batch, tokenizer, max_length):
    return tokenizer(batch["text"], max_length=max_length, truncation=True)

def preprocess_dataset(tokenizer, max_length, seed, dataset):
    dataset = dataset.map(create_prompt_formats)   # add "text"
    _preprocessing_function = partial(preprocess_batch, max_length=max_length, tokenizer=tokenizer)
    dataset = dataset.map(_preprocessing_function, batched=True,
                          remove_columns=["id", "topic", "dialogue", "summary"])
    dataset = dataset.filter(lambda sample: len(sample["input_ids"]) < max_length)   # drop too-long samples
    return dataset.shuffle(seed=seed)

train_dataset = preprocess_dataset(tokenizer, max_length, seed, dataset["train"])
eval_dataset = preprocess_dataset(tokenizer, max_length, seed, dataset["validation"])
print(train_dataset, eval_dataset)

## 7. LoRA

In [ ]:
config = LoraConfig(
    r=64,                    # rank: size of the small matrices A and B
    lora_alpha=16,           # scaling factor: LoRA output is multiplied by alpha / r
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],   # attention projections
    lora_dropout=0.01,
    bias="none",
    task_type="CAUSAL_LM",
)

# Skip prepare_model_for_kbit_training: it casts the large embedding/lm_head (128k vocab) to fp32,
# which does not fit in T4 memory. Do only the parts we need manually.
original_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
original_model.enable_input_require_grads()   # needed for gradient checkpointing with frozen weights
peft_model = get_peft_model(original_model, config)
peft_model.print_trainable_parameters()
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## 8. Training

In [ ]:
peft_training_args = TrainingArguments(
    output_dir="/kaggle/temp/checkpoints",   # keep checkpoints out of the saved output
    per_device_train_batch_size=1,     # fits in T4 memory
    per_device_eval_batch_size=1,      # eval default is 8, which causes OOM
    gradient_accumulation_steps=8,     # effective batch size = 8
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_steps=10,
    optim="paged_adamw_8bit",          # memory-efficient optimizer
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    group_by_length=True,              # batch samples of similar length to reduce padding
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=1,
    report_to="none",
)

peft_model.config.use_cache = False   # cache is only useful for generation

peft_trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=peft_training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [ ]:
torch.cuda.empty_cache()
train_result = peft_trainer.train()
print(train_result)

## 9. Save

In [ ]:
os.makedirs("/kaggle/working/results", exist_ok=True)

adapter_dir = "/kaggle/working/llama3-dialogsum-qlora-adapter"
peft_model.save_pretrained(adapter_dir)   # only the LoRA weights, not the 16GB base model
tokenizer.save_pretrained(adapter_dir)
!ls -lh {adapter_dir}

log_df = pd.DataFrame(peft_trainer.state.log_history)
log_df.to_csv("/kaggle/working/results/training_log.csv", index=False)
print(log_df[["step", "loss", "eval_loss"]].groupby("step").first())

del peft_trainer
gc.collect()
torch.cuda.empty_cache()

## 10. Evaluation after fine-tuning

In [ ]:
peft_model.eval()
peft_model.config.use_cache = True

finetuned_scores, finetuned_preds = evaluate_rouge(peft_model)

comparison = pd.DataFrame({"baseline": baseline_scores, "fine-tuned": finetuned_scores})
comparison["improvement"] = comparison["fine-tuned"] - comparison["baseline"]
print(comparison)

with open("/kaggle/working/results/rouge_scores.json", "w") as f:
    json.dump({"n_eval": N_EVAL, "baseline": baseline_scores, "fine_tuned": finetuned_scores}, f, indent=2)

pd.DataFrame({
    "dialogue": eval_set["dialogue"],
    "reference": eval_set["summary"],
    "baseline": baseline_preds,
    "fine_tuned": finetuned_preds,
}).to_csv("/kaggle/working/results/predictions.csv", index=False)

In [ ]:
# Example
for i in [10, 20]:
    print("=" * 80)
    print("DIALOGUE:\n", eval_set["dialogue"][i])
    print("\nREFERENCE:\n", eval_set["summary"][i])
    print("\nBASELINE:\n", baseline_preds[i])
    print("\nFINE-TUNED:\n", finetuned_preds[i])